# **Preparation Notebook**



---
## Setup Environment

In [1]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT1",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 36.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hdbscan 0.8.41 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
umap-learn 0.5.11 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
Mounted at /content/gdrive

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT1/data


---
## Student Information

In [2]:
student_name = "Rose Marie Tazbaz"
student_id = "25742507"

In [3]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [4]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [5]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LassoCV
from sklearn.impute import SimpleImputer

### 0.b Import Packages

In [6]:
# DO NOT MODIFY THE CODE IN THIS CELL
import pandas as pd
import altair as alt

---
## A. Feature Selection


In [7]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load training data
try:
  training_df = pd.read_csv(at.folder_path / "vehicles_price_training.csv")
  validation_df = pd.read_csv(at.folder_path / "vehicles_price_validation.csv")
  testing_df = pd.read_csv(at.folder_path / "vehicles_price_testing.csv")
except Exception as e:
  print(e)

### A.1 Approach 1

In [8]:
# Approach 1: R^2 for Feature Importance
# First, drop unnecesary features and check existance of NaN or missing values

# Missing values

cols_to_check = training_df.drop(columns=[
    'vehicle_description','prefix','first_name','last_name','gender',
    'phone_number','secondary_address','building_number','street_name',
    'street_suffix','body_type'
]).columns

nan_counts = training_df[cols_to_check].isna().sum()

print("Number of missing values per column:\n")
print(nan_counts[nan_counts > 0])

rows_with_nan = training_df[cols_to_check][training_df[cols_to_check].isna().any(axis=1)]

print(f"\nNumber of rows with at least one NaN: {rows_with_nan.shape[0]}")
print(rows_with_nan.head(10))

df_clean = training_df.copy()


# Variables with formatting issues

# Doors
df_clean['doors'] = df_clean['doors'].astype(str).str.extract('(\d+)')
df_clean['doors'] = pd.to_numeric(df_clean['doors'], errors='coerce')

# Seats
df_clean['seats'] = df_clean['seats'].astype(str).str.extract('(\d+)')
df_clean['seats'] = pd.to_numeric(df_clean['seats'], errors='coerce')

# Engine capacity
df_clean['engine_capacity'] = df_clean['engine_capacity'].astype(str).str.extract('(\d+\.\d+)')
df_clean['engine_capacity'] = pd.to_numeric(df_clean['engine_capacity'], errors='coerce')

# Fuel consumption
df_clean['fuel_consumption'] = df_clean['fuel_consumption'].astype(str).str.extract('(\d+\.\d+)')
df_clean['fuel_consumption'] = pd.to_numeric(df_clean['fuel_consumption'], errors='coerce')

# Engine cylinders (e.g. "4 cyl")
df_clean['engine_cylinders'] = df_clean['engine_cylinders'].astype(str).str.extract('(\d+)')
df_clean['engine_cylinders'] = pd.to_numeric(df_clean['engine_cylinders'], errors='coerce')


# Missing values

# Fill vehicle_type with mode
most_common_type = df_clean['vehicle_type'].mode()[0]
df_clean['vehicle_type'] = df_clean['vehicle_type'].fillna(most_common_type)

# Convert price to numeric
df_clean['price'] = pd.to_numeric(df_clean['price'], errors='coerce')

# Remove rows with missing target
df_clean = df_clean.dropna(subset=['price'])

# Extract territory from location
df_clean['territory'] = df_clean['location'].str.split(',').str[-1].str.strip()


# Unnecessary columns

target_name = 'price'

features = df_clean.drop(columns=[
    target_name,
    'vehicle_description','prefix','first_name','last_name','gender',
    'phone_number','secondary_address','building_number','street_name',
    'street_suffix','body_type','email','location'
]).copy()


# Encoding for catgoeical values

for col in features.select_dtypes(include='object').columns:
    le = LabelEncoder()
    features[col] = le.fit_transform(features[col].astype(str))


X = features
y = df_clean[target_name]

# Remove rows with remaining NaN
X = X.dropna()
y = y.loc[X.index]


# R2
remaining_features = list(X.columns)
selected_features = []

results = []

model = LinearRegression()

while len(remaining_features) > 0:

    best_feature = None
    best_score = -np.inf

    for feature in remaining_features:

        candidate_features = selected_features + [feature]

        scores = cross_val_score(
            model,
            X[candidate_features],
            y,
            cv=5,
            scoring='r2'
        )

        score = scores.mean()

        if score > best_score:
            best_score = score
            best_feature = feature

    selected_features.append(best_feature)
    remaining_features.remove(best_feature)

    results.append({
        "feature_added": best_feature,
        "R2_score": best_score
    })


# Results

r2_results = pd.DataFrame(results)

print("R2 RESULTS")
print(r2_results)


Number of missing values per column:

vehicle_type       2
doors           1036
seats           1103
price              1
dtype: int64

Number of rows with at least one NaN: 1104
                           email  vehicle_brand  manufacturing_year  \
22        qpatterson@example.com         Subaru              2015.0   
34           ythomas@example.net  Mercedes-Benz              2015.0   
38  hendrickscynthia@example.net         Toyota              2017.0   
41             ana61@example.org         Holden              2016.0   
43          rodney25@example.net         Nissan              2001.0   
62       jessicaking@example.org          Mazda              2010.0   
63     hopkinsandrew@example.org         Toyota              2010.0   
67          mariah83@example.com         Subaru              2015.0   
72            cperez@example.net         Toyota              2016.0   
81       hillstephen@example.com         Toyota              2011.0   

   model_name                   vehicle

In [9]:
feature_selection_1_insights = "In this first approach, I used linear regression and R2 to identify which variables are most useful for predicting the resale price of used cars in Australia. Before applying the feature selection method, I did data cleaning and transformation steps, because some variables in the dataset contained numerical information stored as text. For example, the variables doors and seats were stored as values such as “4 doors” or “5 seats”. In these cases, the numerical value was converted into numeric format so that the model could use it as input. The same happened with variables such as engine_capacity, fuel_consumption, and engine_cylinders, that contained mixed text formats.The numerical parts of these values were extracted and converted. Also, there was a small amount of missing data. The variable vehicle_type had a few missing values, which were replaced with the most common category in the dataset (mode). The target variable price was converted to numeric format, and rows with invalid values were removed. This step ensures that the regression model can perform calculations correctly because “POA” doesn’t say anything about a possible price, so even if there are lots of rows with that value as price, it doesn’t say anything, hence, it’s not relevant. Some variables were unrelated to the pricing problem and they were removed, such as personal information about the owner (for example name, phone number, address and email). These variables do not influence the market value of a vehicle and were excluded for both privacy reasons and model relevance. A new variable called territory was also created by extracting the state or territory from the location field. This feature may capture potential regional differences in car prices across Australia. After cleaning and preparing the dataset, I applied R2. In this case, higher R2 value means the model is better able to explain the variation in car prices. The results show that the first variables selected by the model were manufacturing_year, doors, fuel_type, vehicle_type, and engine_capacity. These features produced the largest changes, meaning they provide the most useful information for explaining vehicle resale prices. From a business perspective, this result makes sense. The manufacturing year reflects the age of the car, which strongly affects resale value. Engine capacity and fuel type also influence the price because they relate to the performance and operating cost of the vehicle. Similarly, vehicle type (for example SUV, sedan, or hatchback) can significantly impact the market price. Other variables like vehicle_brand, kilometres_driven, drive_type, and model_name also contributed to improving the model performance, although with a smaller impact. These features capture additional differences between vehicles, such as brand reputation, usage of the car, and specific model characteristics. From a business scope, this indicates that the resale value of a car is mainly determined by a combination of vehicle age, technical specifications, brand, and usage (kilometres driven). Other characteristics, such as colour or seating capacity, appear to have very little influence on the price."

In [10]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_1_insights', value=feature_selection_1_insights)

### A.2 Approach 2

In [11]:
# Approach 2: Lasso Feature Selection

df_clean = training_df.copy()

# Clean 'doors' and 'seats'
for col in ['doors', 'seats']:
    df_clean[col] = df_clean[col].astype(str).str.extract('(\d+)')
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Fill missing vehicle_type with most frequen value
most_common_type = df_clean['vehicle_type'].mode()[0]
df_clean['vehicle_type'] = df_clean['vehicle_type'].fillna(most_common_type)

# Clean price and drop rows with POA
df_clean['price'] = pd.to_numeric(df_clean['price'], errors='coerce')
df_clean = df_clean.dropna(subset=['price'])

# Extract territory from location
df_clean['territory'] = df_clean['location'].str.split(',').str[-1].str.strip()

# Prepare features
target_name = 'price'

features = df_clean.drop(columns=[
    target_name, 'vehicle_description','prefix','first_name','last_name','gender',
    'phone_number','secondary_address','building_number','street_name','street_suffix',
    'body_type','email','location'
]).copy()

# Encode categorical features
for col in features.select_dtypes(include='object').columns:
    le = LabelEncoder()
    features[col] = le.fit_transform(features[col].astype(str))

X = features
y = df_clean[target_name]

# Handle missing values
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# Train Lasso with cross-validation
lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso.fit(X_scaled, y)

# Extract feature importance
lasso_results = pd.DataFrame({
    "feature": X.columns,
    "coefficient": lasso.coef_
})

lasso_results["importance"] = np.abs(lasso_results["coefficient"])
lasso_results = lasso_results.sort_values(by="importance", ascending=False)

print("Lasso Feature Importance:\n")
print(lasso_results)

Lasso Feature Importance:

               feature   coefficient    importance
6      engine_capacity  14319.896253  14319.896253
12    engine_cylinders -11545.001135  11545.001135
1   manufacturing_year   3607.836524   3607.836524
3         vehicle_type  -2519.688774   2519.688774
10   kilometres_driven   2361.647748   2361.647748
13               doors  -2320.890417   2320.890417
8            fuel_type  -2259.982448   2259.982448
9     fuel_consumption   1001.199165   1001.199165
7           drive_type   -617.561521    617.561521
5    transmission_type   -449.968731    449.968731
14               seats   -423.662412    423.662412
4    vehicle_condition   -349.279240    349.279240
0        vehicle_brand    298.004285    298.004285
15           territory   -270.755049    270.755049
11      vehicle_colour   -159.473920    159.473920
2           model_name     -5.798594      5.798594


In [12]:
feature_selection_2_insights = "In the second approach, I used Lasso (L1 regularization) regression. The results show that engine_capacity is the most influential variable in the model, followed by engine_cylinders, manufacturing_year, and vehicle_type. From a business perspective, this result is reasonable. Engine capacity and the number of cylinders are closely related to the performance and power of a vehicle, which often increases its market value. The manufacturing year represents the age of the car, which is one of the main factors affecting resale price. Newer vehicles usually retain more value in the market. Other variables such as kilometres_driven, doors, and fuel_type also have relatively strong coefficients. This suggests that the usage of the vehicle and its technical configuration also play an important role in determining resale value. For example, cars with lower mileage often have higher resale prices because they are perceived as less worn (better shape). Additional variables such as fuel_consumption, drive_type, and transmission_type have smaller coefficients but still impact the model. These variables capture differences in driving experience and operating costs, which may influence buyer preferences. Some features such as vehicle_colour, territory, and especially model_name have very small coefficients, meaning they provide a very limited impact in this model. This suggests that these variables do not strongly explain variations in resale price when compared to other technical characteristics of the vehicle. When comparing the results from Lasso regression with the previous R2 approach, many variables appear consistently important in both methods. These include: manufacturing_year, engine_capacity, vehicle_type, kilometres_driven, fuel_type and doors. This consistency between both approaches suggests that these variables are strong predictors of vehicle resale price and should be included in the final model. Other variables such as drive_type, transmission_type, and engine_cylinders also appear relevant in the Lasso results and provide additional technical information about the vehicle, so they can also be considered useful predictors. On the other hand, variables like vehicle_colour, vehicle_condition, and model_name appear to have very low importance in the Lasso model and provided little improvement in the R2 approach. From a business perspective, these variables are less directly related to the economic value of the car, so they were excluded to reduce noise and simplify the model. Based on the results of both feature selection approaches, the final set of features selected for the predictive models includes: manufacturing_year, engine_capacity, engine_cylinders, kilometres_driven, vehicle_type, vehicle_brand, fuel_type, fuel_consumption, drive_type, transmission_type, doors, seats and territory. These variables capture the most important aspects influencing resale price, including vehicle age, mechanical specifications, usage, brand reputation, and regional market differences."

In [13]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_2_insights', value=feature_selection_2_insights)

### A.n Approach "\<describe_approach_here\>"

> You can add more cells related to other approaches in this section

In [14]:
# <Student to fill this section and then remove this comment>

In [15]:
# <Student to fill this section>
feature_selection_n_insights = """
provide an explanation on why you use this approach for feature selection and describe its results
"""

In [16]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_n_insights', value=feature_selection_n_insights)

### A.z Final Selection of Features

In [17]:
features_list = ["vehicle_brand","manufacturing_year","model_name","vehicle_type","engine_capacity","fuel_type","fuel_consumption","drive_type","transmission_type","kilometres_driven","engine_cylinders","doors","seats","territory","price"]

In [18]:
feature_selection_explanations = "The criteria I used for feature selections has 3 main conditions: first, it has to be consisten with both approaches, has to be relevant for the business problem (price) and I want to avoid not relevant or redundant variables. I chose to keep these main pricing features; manufacturing_year (vehicle age), engine_capacity (engine affects value), kilometres_driven (vehicle usage) and engine_cylinders (power). I also decided to include vehicle_brand (brand reputation), model_name (trust among users), vechicle_type (usually buyers look for a specific type according to their needs), fuel_type (preference), fuel_consumption (for efficiency), drive_type and transmission_type (driver's prefference). There are three variables that may not be too relevant, but could help find patterns would be doors (how practical the car is), seats (car's capacity) and territory (if there's a geographical factor affecting the prefference or budget). I decided to drop vehicle_colour as it doesn't seem to be important in none of the models, vehicle_condition as there are mainly used cars so it doesn't add much information. The other features were dropped because of privacy reasons and because they were overlapping with other features. Overall, this final feature set captures the most important aspects influencing vehicle resale prices, including vehicle age, technical specifications, usage, brand reputation, and potential regional differences."

In [19]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## B. Data Cleaning

In [20]:
# First, we have to add the new variable "Territory" that is replacing "Location" from all datasets
training_df['territory'] = training_df['location'].str.split(',').str[-1].str.strip()
validation_df['territory'] = validation_df['location'].str.split(',').str[-1].str.strip()
testing_df['territory'] = testing_df['location'].str.split(',').str[-1].str.strip()

In [21]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets
try:
  training_df_clean = training_df[features_list].copy()
  validation_df_clean = validation_df[features_list].copy()
  testing_df_clean = testing_df[features_list].copy()
except Exception as e:
  print(e)

### B.1 Fixing "Price (POA)"

In [22]:
for df in [training_df_clean, validation_df_clean, testing_df_clean]:
    df['price'] = pd.to_numeric(df['price'], errors='coerce')

training_df_clean = training_df_clean.dropna(subset=['price'])
validation_df_clean = validation_df_clean.dropna(subset=['price'])
testing_df_clean = testing_df_clean.dropna(subset=['price'])

In [23]:
# <Student to fill this section and then remove this comment>
data_cleaning_1_explanations = "The target variable of the model is “price”, so it has to be stored as a numeric value. During the data exploration phase, I observed that some rows contained the value “POA” (Price on application) instead of a numeric price. This creates a problem because regression models need numerical values to learn the relationship between the features and the target variable.To fix this, the price column was converted to a numeric format. When converting the values, any non-numeric entries such as “POA” were automatically converted to missing values (NaN). These rows were then removed from the dataset. Even though there were several rows containing “POA”, I still dropped them because “POA” doesn’t really provide any useful information about the actual price of the vehicle, and as the objective of the project is to predict a numeric resale price, keeping these observations would not help the model learn meaningful patterns. Removing them ensures that the model is trained only on valid numerical price values, which improves the reliability of the predictions."

In [24]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### B.2 Fixing "doors"

In [25]:
for df in [training_df_clean, validation_df_clean, testing_df_clean]:
    df['doors'] = df['doors'].astype(str).str.extract('(\d+)')
    df['doors'] = pd.to_numeric(df['doors'], errors='coerce')

In [26]:
data_cleaning_2_explanations = "The variable “doors” contained values stored as text (“4 doors”). This creates a problem because when using models, they expect numerical values when a feature represents a quantity. In this case, the number of doors is a numerical feature of the vehicle and should be treated as a numeric variable. To fix this issue, the numeric part of the value was extracted from the text and converted into a numeric format. For example, the value “4 doors” was transformed into the number 4. Fixing this variable is important because the number of doors may influence how practical a vehicle is for different users. For example, cars with more doors may be more convenient for families or passengers, which could affect the resale price. Converting the variable into numeric format allows the model to correctly interpret and use this information when predicting vehicle prices."

In [27]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### B.3 Fixing "seats"

In [28]:
for df in [training_df_clean, validation_df_clean, testing_df_clean]:
    df['seats'] = df['seats'].astype(str).str.extract('(\d+)')
    df['seats'] = pd.to_numeric(df['seats'], errors='coerce')

In [29]:
data_cleaning_3_explanations = "The variable “seats” had a similar issue to the “doors” variable. The values were stored as text, such as “5 seats”, which combines both numeric and textual information, and it should be treated as a numeric variable. To solve this issue, the numeric part of the value was extracted and converted into a numeric format. For example, the value “5 seats” was transformed into the number 5. This transformation is important because the seating capacity can influence the practicality of a vehicle and the type of buyers interested in it. For example, larger cars with more seats may appeal to families, while smaller cars may target individual drivers. Converting this variable to numeric format allows the model to use this information when predicting resale prices."

In [30]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

### B.4 Fixing "Engine capacity"

In [31]:
for df in [training_df_clean, validation_df_clean, testing_df_clean]:
    df['engine_capacity'] = df['engine_capacity'].astype(str).str.extract('(\d+\.\d+)')
    df['engine_capacity'] = pd.to_numeric(df['engine_capacity'], errors='coerce')

In [32]:
data_cleaning_4_explanations = "The variable “engine_capacity” contained mixed text values such as “4 cyl, 2.0 L”. This format combines two different pieces of information: the number of cylinders and the engine size in liters. Since engine capacity is a quantitative technical specification, it should be represented as a numeric value. To fix this issue, the numeric engine size (in liters) was extracted from the text and converted into a numeric format. This allows the model to interpret the engine size as a continuous variable. This transformation is important because engine capacity is strongly related to vehicle performance and power. In the used car market, vehicles with larger engines often have higher prices, while smaller engines may appeal to buyers looking for better fuel efficiency. By converting this feature into a numeric variable, the model can better learn the relationship between engine size and resale price."

In [33]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_4_explanations', value=data_cleaning_4_explanations)

### B.5 Fixing "fuel consumption"

In [34]:
for df in [training_df_clean, validation_df_clean, testing_df_clean]:
    df['fuel_consumption'] = df['fuel_consumption'].astype(str).str.extract('(\d+\.\d+)')
    df['fuel_consumption'] = pd.to_numeric(df['fuel_consumption'], errors='coerce')

In [35]:
data_cleaning_5_explanations = "The variable “fuel_consumption” contained values stored as text with units, such as “7.5 L/100km”. This format is impossible to be understood by machine learning models from interpreting the value as a numeric variable. To address this issue, the numeric fuel consumption value was extracted and converted into numeric format. For example, “7.5 L/100km” was transformed into the numeric value 7.5. This step is important because fuel efficiency can influence a buyer’s decision when purchasing a used vehicle. Cars with lower fuel consumption may be more attractive due to lower operating costs. Converting this variable to numeric format allows the model to analyze how fuel efficiency relates to resale price."

In [36]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_5_explanations', value=data_cleaning_5_explanations)

### B.6 Fixing "engine cylinders"

In [37]:
for df in [training_df_clean, validation_df_clean, testing_df_clean]:
    df['engine_cylinders'] = df['engine_cylinders'].astype(str).str.extract('(\d+)')
    df['engine_cylinders'] = pd.to_numeric(df['engine_cylinders'], errors='coerce')

In [38]:
data_cleaning_6_explanations = "The variable “engine_cylinders” also contained text values such as “4 cyl”. Since the number of cylinders represents a numeric engine feature, it should be stored as a numeric variable. To correct this issue, the number was extracted from the text and converted into numeric format. For example, the value “4 cyl” was transformed into the number 4. This transformation allows the model to use engine configuration as a quantitative feature. The number of cylinders is often related to vehicle power and performance, which can influence the resale price of a car."

In [39]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_6_explanations', value=data_cleaning_6_explanations)

### B.7 Fixing "missing values in vehicle type"

In [40]:
most_common_type = training_df_clean['vehicle_type'].mode()[0]

for df in [training_df_clean, validation_df_clean, testing_df_clean]:
    df['vehicle_type'] = df['vehicle_type'].fillna(most_common_type)

In [41]:
data_cleaning_7_explanations = "During the data inspection process, I noticed that the variable “vehicle_type” contained a small number of missing values. As vehicle type is an important categorical feature describing the class of the car (such as SUV, sedan, or hatchback), it is useful for predicting vehicle prices.To address this issue, missing values were replaced with the most common category (mode) in the dataset. This approach allows the dataset to retain more observations while maintaining a reasonable assumption about the missing values. This step is important because removing these observations could unnecessarily reduce the size of the dataset. By filling the missing values with the most frequent category, the model can continue to use this feature without introducing large distortions in the data."

In [42]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_7_explanations', value=data_cleaning_7_explanations)

---
## C. Feature Engineering

In this section I created new variables that may help the model better capture patterns related to the resale price of vehicles. These new features are derived from existing variables and aim to represent important aspects. From a business perspective, these factors can influence the value of used cars in the market and therefore may improve the predictive performance of the model.

In [43]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  training_df_eng = training_df_clean.copy()
  validation_df_eng = validation_df_clean.copy()
  testing_df_eng = testing_df_clean.copy()
except Exception as e:
  print(e)

### C.1 New Feature "Territory"



In [44]:
# Code was executed in the approach section

In [45]:
feature_engineering_1_explanations = "The first engineered feature is territory. The original dataset contained a location variable with detailed information about the suburb and state where the vehicle was being sold. However, using the full location information would create many unique categories (because of all the different suburbs), making it hard to group by location and eventually infere something. To simplify this information, a new variable called territory was created by extracting only the state or territory from the location field. This allows the model to capture possible regional differences in vehicle prices across Australia. From a business perspective, vehicle prices may vary depending on the region due to differences in demand, supply, and economic conditions."

In [46]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### C.2 New Feature "Engine power (in categories)"



In [47]:
datasets = [training_df_eng, validation_df_eng, testing_df_eng]
for df in datasets:
    df["engine_power_category"] = pd.cut(
        df["engine_capacity"],
        bins=[0, 1.6, 2.5, 10],
        labels=["small_engine", "medium_engine", "large_engine"]
    )

In [48]:
feature_engineering_2_explanations = "The feature engine_power_category was created by grouping engine capacity values into categories. Engine capacity can be very different across vehicles, so grouping them into categories may help the model identify patterns more easily. From a business perspective, vehicles with larger engines are often associated with higher performance and may belong to more expensive market segments. Categorizing engine capacity may therefore help the model better represent differences between vehicle classes."

In [49]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### C.3 New Feature "high mileage"



In [50]:
# Verifying if kilometres_driven is numeric
for df in datasets:
    df["kilometres_driven"] = pd.to_numeric(df["kilometres_driven"], errors="coerce")

# Mileage boundary
mileage_bound = training_df_eng["kilometres_driven"].quantile(0.75)

# High mileage vehicle
for df in datasets:
    df["high_mileage_vehicle"] = (df["kilometres_driven"] > mileage_bound).astype(int)

In [51]:
feature_engineering_3_explanations = "The feature high_mileage_vehicle was created to identify vehicles that have unusually high mileage compared to typical usage. This variable was generated by checking whether the kilometres driven are above the 75th percentile of the dataset. From a business perspective, vehicles with very high mileage are often perceived as more worn and therefore may have lower resale value."

In [52]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_3_explanations', value=feature_engineering_3_explanations)

### C.4 Fixing "efficiency"

In [53]:
for df in datasets:
    df["efficiency"] = pd.cut(
        df["fuel_consumption"],
        bins=[0, 7, 10, 20],
        labels=["efficient", "moderate", "high_consumption"]
    )

In [54]:
# <Student to fill this section and then remove this comment>
feature_engineering_n_explanations = "Another engineered feature is efficiency, which groups vehicles according to their fuel consumption levels. Fuel consumption values were divided into three categories representing efficient, moderate, and high consumption vehicles. From a business perspective, fuel efficiency can influence buyer preferences because it directly affects operating costs. Cars with lower fuel consumption may therefore be more attractive in the used car market."

In [55]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_n_explanations', value=feature_engineering_n_explanations)

In [56]:
training_df_eng.columns

Index(['vehicle_brand', 'manufacturing_year', 'model_name', 'vehicle_type',
       'engine_capacity', 'fuel_type', 'fuel_consumption', 'drive_type',
       'transmission_type', 'kilometres_driven', 'engine_cylinders', 'doors',
       'seats', 'territory', 'price', 'engine_power_category',
       'high_mileage_vehicle', 'efficiency'],
      dtype='object')

---
## D. Data Preparation for Modeling

In [57]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  X_train = training_df_eng.copy()
  X_val = validation_df_eng.copy()
  X_test = testing_df_eng.copy()

  y_train = X_train.pop(target_name)
  y_val = X_val.pop(target_name)
  y_test = X_test.pop(target_name)
except Exception as e:
  print(e)

The target variable (price) was separated from the predictor variables. The remaining variables form the feature matrix (X), which will be used as input to the machine learning models. The price variable represents the output variable (Y), which the models will attempt to predict.

### D.1 Check consistency


In [58]:
print("Training set shape:", X_train.shape)
print("Validation set shape:", X_val.shape)
print("Testing set shape:", X_test.shape)

X_train.head()

Training set shape: (8767, 17)
Validation set shape: (2614, 17)
Testing set shape: (2606, 17)


,vehicle_brand,manufacturing_year,model_name,vehicle_type,engine_capacity,fuel_type,fuel_consumption,drive_type,transmission_type,kilometres_driven,engine_cylinders,doors,seats,territory,engine_power_category,high_mileage_vehicle,efficiency
0,Honda,2014.0,Jazz,Hatchback,1.3,Hybrid,4.5,Front,Automatic,38229.0,4.0,5.0,5.0,NSW,small_engine,0,efficient
1,Ford,2013.0,Ranger,Ute / Tray,3.2,Diesel,9.2,4WD,Automatic,133168.0,5.0,2.0,2.0,QLD,large_engine,0,moderate
2,Porsche,1978.0,911,Convertible,NaN,Leaded,NaN,Rear,Manual,145600.0,6.0,2.0,4.0,NSW,NaN,0,NaN
3,Honda,2006.0,Accord,Sedan,NaN,Unleaded,10.6,Front,Automatic,209508.0,6.0,4.0,5.0,NSW,NaN,1,high_consumption
4,Kia,2012.0,Sorento,SUV,2.2,Diesel,7.3,AWD,Automatic,134686.0,4.0,4.0,7.0,VIC,medium_engine,0,moderate


In [59]:
# <Student to fill this section and then remove this comment>
data_transformation_1_explanations = "Quickly verifying this, to confirm that the datasets were correctly structured. This ensures that the training, validation, and testing sets contain the expected number of observations and variables before exporting them for model training."

In [60]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

### D.2 Data Transformation: Conversion of variables (format)

In [61]:
data_transformation_2_explanations = "Many variables in the dataset contained numerical information stored as text. For example, the variables doors and seats included values such as “4 doors” or “5 seats”. Similarly, engine_capacity, engine_cylinders, and fuel_consumption contained values with mixed formats that combined numbers and units. In these cases, the numerical values were extracted from the text and converted into numeric format."

In [62]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_2_explanations', value=data_transformation_2_explanations)

### D.3 Data Transformation: Missing values


In [63]:
# <Student to fill this section and then remove this comment>
data_transformation_3_explanations = "Some variables contained missing values.  For example, the variable vehicle_type had a few missing entries.  These values were replaced using the most frequent category (mode) in the dataset to maintain dataset consistency and minimizing a potential bias."

In [64]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_3_explanations', value=data_transformation_3_explanations)

### D.4 Data Transformation: Cleaning the target variable


In [65]:
# <Student to fill this section and then remove this comment>
data_transformation_n_explanations = "The target variable price originally contained non-numeric values such as “POA” (Price on Application).  Since regression models require numeric target values, these entries were converted to missing values and removed from the dataset.  This step ensures that the models can correctly learn numerical relationships between the features and the predicted price."

In [66]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_n_explanations', value=data_transformation_n_explanations)

---
## E. Save Datasets

> Do not change this code

In [67]:
# DO NOT MODIFY THE CODE IN THIS CELL

try:
  X_train.to_csv(at.folder_path / 'X_train.csv', index=False)
  y_train.to_csv(at.folder_path / 'y_train.csv', index=False)

  X_val.to_csv(at.folder_path / 'X_val.csv', index=False)
  y_val.to_csv(at.folder_path / 'y_val.csv', index=False)

  X_test.to_csv(at.folder_path / 'X_test.csv', index=False)
  y_test.to_csv(at.folder_path / 'y_test.csv', index=False)
except Exception as e:
  print(e)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=fbbefce8-41ae-47c6-bc64-96decd566c0b' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>